# 4 – Backend-Plugin: Platform Status API

Dieses Notebook erstellt ein Backend-Plugin `platform-status`. Der Endpunkt liefert technische Laufzeitinformationen, die für Diagnose, Kubernetes-Probes oder ein späteres Frontend-Widget verwendet werden können.

Bereitgestellte Endpunkte:

- `GET /api/platform-status/health`
- `GET /api/platform-status/info`


> **Voraussetzung:** Die Backstage-Installation liegt unter `~/mybackstage`.
>
> Shell-Zellen werden über Python mit `subprocess` ausgeführt. Vor Änderungen wird jeweils eine Sicherung angelegt.

In [ ]:
from pathlib import Path
ROOT = Path.home() / "mybackstage"
assert ROOT.exists(), f"{ROOT} wurde nicht gefunden"
PLUGIN = ROOT / "plugins/platform-status-backend"
print("Backstage:", ROOT)

## Backend-Plugin erzeugen

Führe `yarn new` aus und wähle:

- **backend-plugin**
- Plugin ID: **platform-status**

In [ ]:
import subprocess
if not PLUGIN.exists():
    subprocess.run(["yarn", "new"], cwd=ROOT, check=True)
else:
    print("Plugin-Verzeichnis existiert bereits:", PLUGIN)

## Router und Plugin implementieren

In [ ]:
from pathlib import Path
import shutil, datetime

src = PLUGIN / "src"
src.mkdir(parents=True, exist_ok=True)

files = {
    src / "router.ts": """import express from 'express';
import Router from 'express-promise-router';
import type { LoggerService } from '@backstage/backend-plugin-api';

export interface RouterOptions {
  logger: LoggerService;
}

export async function createRouter(
  options: RouterOptions,
): Promise<express.Router> {
  const { logger } = options;
  const router = Router();
  router.use(express.json());

  router.get('/health', (_req, res) => {
    res.json({
      status: 'ok',
      plugin: 'platform-status',
      timestamp: new Date().toISOString(),
    });
  });

  router.get('/info', (_req, res) => {
    res.json({
      plugin: 'platform-status',
      nodeVersion: process.version,
      environment: process.env.NODE_ENV ?? 'development',
      uptimeSeconds: Math.round(process.uptime()),
      hostname: process.env.HOSTNAME ?? 'local',
    });
  });

  logger.info('Platform Status routes registered');
  return router;
}
""",
    src / "plugin.ts": """import {
  coreServices,
  createBackendPlugin,
} from '@backstage/backend-plugin-api';
import { createRouter } from './router';

export const platformStatusPlugin = createBackendPlugin({
  pluginId: 'platform-status',
  register(env) {
    env.registerInit({
      deps: {
        logger: coreServices.logger,
        httpRouter: coreServices.httpRouter,
      },
      async init({ logger, httpRouter }) {
        httpRouter.use(await createRouter({ logger }));
        httpRouter.addAuthPolicy({
          path: '/health',
          allow: 'unauthenticated',
        });
        httpRouter.addAuthPolicy({
          path: '/info',
          allow: 'unauthenticated',
        });
      },
    });
  },
});
""",
    src / "index.ts": """export { platformStatusPlugin as default } from './plugin';
export { createRouter } from './router';
""",
}

for path, content in files.items():
    if path.exists():
        backup = path.with_suffix(path.suffix + f".bak-{datetime.datetime.now():%Y%m%d-%H%M%S}")
        shutil.copy2(path, backup)
    path.write_text(content)
    print("geschrieben:", path.relative_to(ROOT))

## Backend registrieren

In [ ]:
from pathlib import Path
import shutil, datetime

index_file = ROOT / "packages/backend/src/index.ts"
backup = index_file.with_suffix(f".ts.bak-{datetime.datetime.now():%Y%m%d-%H%M%S}")
shutil.copy2(index_file, backup)

text = index_file.read_text()
line = "backend.add(import('@internal/plugin-platform-status-backend'));"

# Der genaue Workspace-Paketname kommt aus package.json.
import json
pkg = json.loads((PLUGIN / "package.json").read_text())
package_name = pkg["name"]
line = f"backend.add(import('{package_name}'));"

if line not in text:
    anchor = "backend.start();"
    if anchor not in text:
        raise RuntimeError("backend.start(); wurde nicht gefunden")
    index_file.write_text(text.replace(anchor, f"{line}\n\n{anchor}"))
    print("Backend-Plugin registriert:", package_name)
else:
    print("Backend-Plugin ist bereits registriert.")

print("Sicherung:", backup)

## TypeScript prüfen

In [ ]:
import subprocess
subprocess.run(["yarn", "tsc"], cwd=ROOT, check=True)

## Endpunkte testen

In [ ]:
print("Backstage zuerst in einem separaten Terminal starten:")
print(f"cd {ROOT} && yarn start")
print()
print("Danach testen:")
print("curl -s http://localhost:7007/api/platform-status/health | jq")
print("curl -s http://localhost:7007/api/platform-status/info | jq")

## Beispielantwort

```json
{
  "status": "ok",
  "plugin": "platform-status",
  "timestamp": "2026-07-22T10:00:00.000Z"
}
```

Für Kubernetes kann der Health-Endpunkt später als HTTP-Probe eingesetzt werden. Für eine produktive Instanz sollte geprüft werden, ob der Endpoint bewusst unauthentifiziert erreichbar sein soll.